In [ ]:
!pip install langchain langchain_community langchain_openai google-search-results streamlit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 975.5/975.5 kB 12.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 43.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 50.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 337.4/337.4 kB 24.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 328.3/328.3 kB 22.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 23.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 11.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 51.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.0/83.0 kB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 4.3 MB/s e

In [ ]:
%%writefile app.py
import os
import streamlit as st
from langchain.agents import AgentType, initialize_agent, load_tools
from langchain.prompts import ChatPromptTemplate
from langchain.output_parsers import ResponseSchema, StructuredOutputParser
from langchain_openai import ChatOpenAI

# Set the environment variables for the API keys
os.environ["SERPAPI_API_KEY"] = "128b35d53e53c6a2544139bef047bbb229c11cb984e091d378c56badea0fb669"
os.environ["OPENAI_API_KEY"] = "sk-proj-Oj5qf0wdo1Rm9GkDOPVCT3BlbkFJ4Iag2xTN4ug8Oertvqza"
#OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
#SERPAPI_API_KEY = os.getenv('SERPAPI_API_KEY')
# Load the tools and initialize the LLM
tools = load_tools(["serpapi"])
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.0)

# Define the response schemas
company_name = ResponseSchema(name="company_name", description="this is the name of the company")
phone_number = ResponseSchema(name="phone_number", description="this is the phone number of the company")
email = ResponseSchema(name="email", description="this is the email of the company")
website = ResponseSchema(name="website", description="this is the website of the company")
address = ResponseSchema(name="address", description="this is the address of the company")
city = ResponseSchema(name="city", description="this is the city of the company")
postal_code = ResponseSchema(name="postal_code", description="this is the postal code of the company")
products = ResponseSchema(name="products", description="this is the products offered by the company")
services = ResponseSchema(name="services", description="this is the services offered by the company")
revenue = ResponseSchema(name="revenue", description="this is the revenue of the company")
competitors = ResponseSchema(name="competitors", description="these are the competitors of the company")
branches = ResponseSchema(name="branches", description="these are the branches of the company")
careers = ResponseSchema(name="careers", description="this is the career page or job offerings of the company")

# Combine the schemas into a structured output parser
response_schema = [
    company_name, phone_number, email, website, address, city, postal_code,
    products, services, revenue, competitors, branches, careers
]
output_parser = StructuredOutputParser.from_response_schemas(response_schema)
format_instruction = output_parser.get_format_instructions()

# Define the prompt template
ts = """
You are an intelligent search master and analyst who can search the internet using the SerpAPI tool and retrieve company information including company name, phone number, email, company website, address, city, postal code, products, services, revenue, competitors, branches, and careers.
Take the input below delimited by triple backticks and use it to search and retrieve the information like company_name,input phone_number,input email,input website, input address, input city, input postal_code,
   input products, input services,input revenue, input competitors, input branches,input careers the data should be upto date and efficient using the SerpAPI tool and wikipedia data.
input:```{input}```
{format_instruction}
"""
prompt = ChatPromptTemplate.from_template(ts)

# Streamlit App
st.title("Company Information Retrieval")

# Create a form for user input and submit button
with st.form(key='company_form'):
    user_input = st.text_input("Enter the company name:").lower()  # Convert input to lowercase
    submit_button = st.form_submit_button(label='Submit')

if submit_button:
    # Format the prompt with user input
    fs = prompt.format_messages(input=user_input, format_instruction=format_instruction)

    # Initialize the agent
    agent = initialize_agent(tools, llm, agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True)

    # Run the agent with the formatted input
    response = agent.run(fs)
    output = output_parser.parse(response)

    # Display the results with enhanced styling
    st.markdown("### Company Information")
    st.markdown(f"**Company Name:** {output.get('company_name', 'N/A')}")
    st.markdown(f"**Phone Number:** {output.get('phone_number', 'N/A')}")
    st.markdown(f"**Email:** {output.get('email', 'N/A')}")
    st.markdown(f"**Website:** {output.get('website', 'N/A')}")
    st.markdown(f"**Address:** {output.get('address', 'N/A')}")
    st.markdown(f"**City:** {output.get('city', 'N/A')}")
    st.markdown(f"**Postal Code:** {output.get('postal_code', 'N/A')}")
    st.markdown(f"**Products:** {output.get('products', 'N/A')}")
    st.markdown(f"**Services:** {output.get('services', 'N/A')}")
    st.markdown(f"**Revenue:** {output.get('revenue', 'N/A')}")
    st.markdown(f"**Competitors:** {output.get('competitors', 'N/A')}")
    st.markdown(f"**Branches:** {output.get('branches', 'N/A')}")
    st.markdown(f"**Careers:** {output.get('careers', 'N/A')}")


Writing app.py


In [ ]:
!wget -q -O - ipv4.icanhazip.com

34.148.189.1


In [ ]:
! streamlit run app.py & npx localtunnel --port 8501




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.148.189.1:8501

npx: installed 22 in 12.451s
your url is: https://cuddly-moons-chew.loca.lt
/usr/local/lib/python3.10/dist-packages/langchain/_api/module_import.py:92: LangChainDeprecationWarning: Importing load_tools from langchain.agents is deprecated. Please replace deprecated imports:

>> from langchain.agents import load_tools

with new imports of:

>> from langchain_community.agent_toolkits.load_tools import load_tools
You can use the langchain cli to **automatically** upgrade many imports. Please see documentation here <https://python.langchain.com/v0.2/docs/versions/v0_2/>
  warn_deprecated(
/usr/local/lib/python3.10/dist-packages/langchain/_api/module_import.py:92: LangChainDeprecationWarning: Importing load_tools from langchain.agents is deprecated. Please replace deprecated imports:

>> from langchain.agents import loa